## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q open_clip_torch matplotlib pillow requests ipywidgets

import torch
import open_clip
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import json
import os
import base64
from pathlib import Path

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Running on: {device}")
print(f"📁 Working directory: {os.getcwd()}")

## 2. Load BiomedCLIP Model

In [ ]:
print("📦 Loading BiomedCLIP model...")
model, preprocess = open_clip.create_model_from_pretrained(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
tokenizer = open_clip.get_tokenizer(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model.to(device)
model.eval()
print("✅ BiomedCLIP loaded successfully!")

## 3. Load TriMedAgent Configuration

In [ ]:
# Load configuration from labels.json
config_path = Path("serve/labels.json")

if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        TRIMED_CONFIG = json.load(f)
    print("✅ Loaded configuration from serve/labels.json")
else:
    # Default configuration
    TRIMED_CONFIG = {
        "triage_labels": [
            "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology",
            "Ultrasound", "Dermoscopy", "Gross pathology", "Bone X-ray", "Lung CT"
        ],
        "gatekeeper_prompts": {
            "positive": "Pathological finding, lesion, tumor, abnormality",
            "negative": "Normal tissue, healthy anatomy, background noise, blurry area"
        },
        "thresholds": {
            "triage_confidence": 0.5,
            "gatekeeper_confidence": 0.6,
            "low_confidence_fallback": 0.3
        },
        "conditional_execution": {
            "enabled": True,
            "skip_detection_modalities": ["Histopathology", "Dermoscopy"],
            "tool_mapping": {
                "Chest X-ray": ["grounding_dino", "medsam"],
                "Brain MRI": ["grounding_dino", "medsam"],
                "default": ["grounding_dino"]
            }
        }
    }
    print("⚠️ Using default configuration")

# Display configuration
print("\n📋 Configuration:")
print(f"  • Triage labels: {len(TRIMED_CONFIG['triage_labels'])} modalities")
print(f"  • Triage threshold: {TRIMED_CONFIG['thresholds']['triage_confidence']}")
print(f"  • Gatekeeper threshold: {TRIMED_CONFIG['thresholds']['gatekeeper_confidence']}")

## 4. Core TriMedAgent Functions

In [ ]:
class TriMedAgent:
    """TriMedAgent - Intelligent Triage Pipeline for Medical Image Analysis
    
    Hỗ trợ cả single-image analysis và chatbot conversation.
    """
    
    def __init__(self, model, preprocess, tokenizer, config, device="cuda"):
        self.model = model
        self.preprocess = preprocess
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        
        # Cache triage result
        self.last_triage_result = None
        
        # ================= CHATBOT STATE =================
        self.conversation_history = []  # Lưu lịch sử chat
        self.current_image = None       # Ảnh hiện tại trong phiên
        self.current_image_b64 = None   # Base64 của ảnh
        self.session_active = False     # Phiên chat đang hoạt động?
        self.detected_boxes = []        # Boxes từ detection (nếu có)
        self.verified_boxes = []        # Boxes đã verify bởi Gatekeeper
    
    def _classify(self, image, labels, template="this is a photo of {}"):
        """Core classification function using BiomedCLIP"""
        image_tensor = self.preprocess(image).unsqueeze(0).to(self.device)
        texts = self.tokenizer([template.format(l) for l in labels]).to(self.device)
        
        with torch.no_grad():
            image_features = self.model.encode_image(image_tensor)
            text_features = self.model.encode_text(texts)
            
            # Normalize
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            
            # Calculate similarity
            probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            probs = probs.cpu().numpy()[0]
        
        # Return all predictions
        all_preds = dict(zip(labels, probs.tolist()))
        top_idx = probs.argmax()
        
        return {
            "prediction": labels[top_idx],
            "confidence": float(probs[top_idx]),
            "all_predictions": all_preds
        }
    
    # ================= STEP 1: TRIAGE (PERCEPTION) =================
    def run_triage(self, image, verbose=True):
        """
        STEP 1: Visual Triage - Classify medical image modality
        """
        if verbose:
            print("\n" + "="*60)
            print("🔬 STEP 1: VISUAL TRIAGE (Perception)")
            print("="*60)
        
        labels = self.config["triage_labels"]
        result = self._classify(image, labels, "this is a {}") 
        
        # Get recommended tools based on modality
        tool_mapping = self.config.get("conditional_execution", {}).get("tool_mapping", {})
        recommended_tools = tool_mapping.get(result["prediction"], tool_mapping.get("default", ["grounding_dino"]))
        
        # Check if should skip detection
        skip_detection = result["prediction"] in self.config.get("conditional_execution", {}).get("skip_detection_modalities", [])
        
        # Build context prompt
        threshold = self.config["thresholds"]["triage_confidence"]
        if result["confidence"] >= threshold:
            context = f"[System Context: Image identified as {result['prediction']} ({result['confidence']:.0%} confidence). Recommended tools: {', '.join(recommended_tools)}]"
        else:
            context = f"[System Context: Image modality unclear (best guess: {result['prediction']}, {result['confidence']:.0%}). Proceed with general analysis.]"
        
        triage_result = {
            "modality": result["prediction"],
            "confidence": result["confidence"],
            "all_predictions": result["all_predictions"],
            "recommended_tools": recommended_tools,
            "skip_detection": skip_detection,
            "context_prompt": context
        }
        
        self.last_triage_result = triage_result
        
        if verbose:
            print(f"\n📊 Triage Result:")
            print(f"   • Detected Modality: {result['prediction']}")
            print(f"   • Confidence: {result['confidence']:.2%}")
            print(f"   • Recommended Tools: {recommended_tools}")
            print(f"   • Skip Detection: {skip_detection}")
            print(f"\n💉 Context Prompt:")
            print(f"   {context}")
            
            print(f"\n📈 Top-5 Predictions:")
            sorted_preds = sorted(result["all_predictions"].items(), key=lambda x: x[1], reverse=True)[:5]
            for i, (label, prob) in enumerate(sorted_preds, 1):
                bar = "█" * int(prob * 20)
                print(f"   {i}. {label}: {prob:.2%} {bar}")
        
        return triage_result
    
    # ================= STEP 2: CONTEXT INJECTION (REASONING) =================
    def inject_context(self, original_prompt, triage_result=None, verbose=True):
        """STEP 2: Inject triage context into prompt"""
        if verbose:
            print("\n" + "="*60)
            print("🧠 STEP 2: CONTEXT INJECTION (Reasoning)")
            print("="*60)
        
        if triage_result is None:
            triage_result = self.last_triage_result
        
        if triage_result is None:
            if verbose:
                print("⚠️ No triage result available. Using original prompt.")
            return original_prompt
        
        context = triage_result["context_prompt"]
        enhanced_prompt = f"{context}\n\n{original_prompt}"
        
        if verbose:
            print(f"\n📝 Original Prompt:")
            print(f"   {original_prompt[:100]}..." if len(original_prompt) > 100 else f"   {original_prompt}")
            print(f"\n✨ Enhanced Prompt:")
            print(f"   {enhanced_prompt[:200]}..." if len(enhanced_prompt) > 200 else f"   {enhanced_prompt}")
        
        return enhanced_prompt
    
    # ================= STEP 3: GATEKEEPER (VERIFICATION) =================
    def run_gatekeeper(self, image, boxes, target_entity="abnormality", verbose=True):
        """STEP 3: Gatekeeper - Verify detected regions to filter false positives"""
        if verbose:
            print("\n" + "="*60)
            print("🛡️ STEP 3: GATEKEEPER VERIFICATION")
            print("="*60)
        
        if not boxes:
            if verbose:
                print("⚠️ No boxes to verify.")
            return {"verified_boxes": [], "rejected_indices": [], "details": []}
        
        w, h = image.size
        threshold = self.config["thresholds"]["gatekeeper_confidence"]
        
        pos_prompt = self.config["gatekeeper_prompts"]["positive"]
        neg_prompt = self.config["gatekeeper_prompts"]["negative"]
        pos_label = f"{pos_prompt} of {target_entity}"
        neg_label = neg_prompt
        labels = [pos_label, neg_label]
        
        if verbose:
            print(f"\n🎯 Target Entity: {target_entity}")
            print(f"📏 Threshold: {threshold}")
            print(f"📦 Boxes to verify: {len(boxes)}")
        
        verified_boxes = []
        rejected_indices = []
        details = []
        
        if verbose:
            print(f"\n🔍 Verifying each box:")
        
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box
            left = max(0, int(x1 * w))
            top = max(0, int(y1 * h))
            right = min(w, int(x2 * w))
            bottom = min(h, int(y2 * h))
            
            crop = image.crop((left, top, right, bottom))
            result = self._classify(crop, labels, "this is {}")
            
            is_positive = result["prediction"] == pos_label
            passed = is_positive and result["confidence"] >= threshold
            
            detail = {
                "box_index": i,
                "box": box,
                "prediction": result["prediction"],
                "confidence": result["confidence"],
                "passed": passed
            }
            details.append(detail)
            
            if passed:
                verified_boxes.append(box)
                status = "✅ KEEP"
            else:
                rejected_indices.append(i)
                status = "❌ DROP"
            
            if verbose:
                print(f"   Box {i}: {status} (conf: {result['confidence']:.2%})")
        
        if verbose:
            print(f"\n📊 Gatekeeper Summary: {len(verified_boxes)}/{len(boxes)} boxes kept")
        
        return {
            "verified_boxes": verified_boxes,
            "rejected_indices": rejected_indices,
            "details": details
        }
    
    # ================= FULL PIPELINE =================
    def run_full_pipeline(self, image, user_question, simulated_boxes=None, target_entity="abnormality"):
        """Run the complete TriMedAgent pipeline"""
        print("\n" + "#"*70)
        print("#" + " "*20 + "TRIMEDAGENT PIPELINE" + " "*20 + "#")
        print("#"*70)
        
        triage_result = self.run_triage(image)
        enhanced_prompt = self.inject_context(user_question, triage_result)
        
        gatekeeper_result = None
        if simulated_boxes:
            if triage_result["skip_detection"]:
                print("\n" + "="*60)
                print("⏭️ CONDITIONAL EXECUTION: Skipping detection for", triage_result["modality"])
                print("="*60)
            else:
                gatekeeper_result = self.run_gatekeeper(image, simulated_boxes, target_entity)
        
        return {
            "triage": triage_result,
            "enhanced_prompt": enhanced_prompt,
            "gatekeeper": gatekeeper_result
        }
    
    # ================= CHATBOT FUNCTIONS =================
    
    def start_new_session(self, image=None):
        """Bắt đầu phiên chat mới"""
        self.conversation_history = []
        self.current_image = image
        self.last_triage_result = None
        self.detected_boxes = []
        self.verified_boxes = []
        self.session_active = True
        
        print("\n" + "🔄"*30)
        print("🆕 NEW CHAT SESSION STARTED")
        print("🔄"*30)
        
        if image is not None:
            # Tự động chạy Triage khi có ảnh
            print("\n📷 Image detected. Running automatic triage...")
            self.last_triage_result = self.run_triage(image, verbose=True)
            
        return "Session started. " + (f"Image classified as: {self.last_triage_result['modality']}" if self.last_triage_result else "No image uploaded.")
    
    def set_image(self, image):
        """Set hoặc thay đổi ảnh trong phiên chat hiện tại"""
        self.current_image = image
        
        # Reset detection results khi đổi ảnh
        self.detected_boxes = []
        self.verified_boxes = []
        
        # Chạy Triage cho ảnh mới
        print("\n📷 New image set. Running triage...")
        self.last_triage_result = self.run_triage(image, verbose=True)
        
        # Thêm vào history
        self.conversation_history.append({
            "role": "system",
            "content": f"[New image uploaded: {self.last_triage_result['modality']} ({self.last_triage_result['confidence']:.0%})]"
        })
        
        return self.last_triage_result
    
    def _generate_response(self, user_message, use_tools=True):
        """
        Sinh response cho user message.
        Trong demo này, chúng ta simulate LLM response.
        Trong production, đây sẽ gọi LLaVA-Med API.
        """
        # Kiểm tra các intent đặc biệt
        message_lower = user_message.lower()
        
        # Intent: Detection/Localization
        if any(word in message_lower for word in ["detect", "find", "locate", "where", "tìm", "phát hiện", "vị trí"]):
            return self._handle_detection_intent(user_message)
        
        # Intent: Segmentation
        if any(word in message_lower for word in ["segment", "outline", "boundary", "phân vùng"]):
            return self._handle_segmentation_intent(user_message)
        
        # Intent: Classification/Diagnosis
        if any(word in message_lower for word in ["what is", "diagnose", "classify", "là gì", "chẩn đoán"]):
            return self._handle_classification_intent(user_message)
        
        # Intent: Description
        if any(word in message_lower for word in ["describe", "explain", "tell me", "mô tả", "giải thích"]):
            return self._handle_description_intent(user_message)
        
        # Default: General Q&A
        return self._handle_general_intent(user_message)
    
    def _handle_detection_intent(self, user_message):
        """Xử lý intent phát hiện vùng bất thường"""
        if self.current_image is None:
            return "❌ No image available. Please upload an image first."
        
        # Simulate detection (trong production sẽ gọi Grounding DINO)
        # Tạo fake boxes dựa trên loại ảnh
        modality = self.last_triage_result.get("modality", "Unknown") if self.last_triage_result else "Unknown"
        
        if "X-ray" in modality:
            # Simulate lung regions
            self.detected_boxes = [
                [0.25, 0.3, 0.45, 0.7],  # Left lung
                [0.55, 0.3, 0.75, 0.7],  # Right lung
                [0.35, 0.25, 0.65, 0.45], # Upper region
            ]
            target = "lung opacity"
        elif "MRI" in modality:
            self.detected_boxes = [
                [0.3, 0.3, 0.7, 0.7],
                [0.4, 0.2, 0.6, 0.4],
            ]
            target = "lesion"
        else:
            self.detected_boxes = [
                [0.2, 0.2, 0.8, 0.8],
            ]
            target = "abnormality"
        
        # Chạy Gatekeeper để verify
        gate_result = self.run_gatekeeper(self.current_image, self.detected_boxes, target, verbose=True)
        self.verified_boxes = gate_result["verified_boxes"]
        
        # Build response
        response = f"""🔍 **Detection Results for {modality}**

**Tool Used:** Grounding DINO + Gatekeeper

**Raw Detections:** {len(self.detected_boxes)} regions found
**After Gatekeeper:** {len(self.verified_boxes)} verified regions

"""
        if self.verified_boxes:
            response += "**Verified Findings:**\n"
            for i, box in enumerate(self.verified_boxes):
                response += f"  • Region {i+1}: [{box[0]:.2f}, {box[1]:.2f}, {box[2]:.2f}, {box[3]:.2f}]\n"
        else:
            response += "No significant abnormalities detected after verification."
        
        return response
    
    def _handle_segmentation_intent(self, user_message):
        """Xử lý intent phân vùng"""
        if self.current_image is None:
            return "❌ No image available. Please upload an image first."
        
        if not self.verified_boxes:
            return "⚠️ Please run detection first to identify regions for segmentation."
        
        response = f"""🎯 **Segmentation Results**

**Tool Used:** Grounding DINO + MedSAM + Gatekeeper

**Segmented Regions:** {len(self.verified_boxes)}

The detected regions have been segmented. In production, MedSAM would generate precise masks for each verified region.
"""
        return response
    
    def _handle_classification_intent(self, user_message):
        """Xử lý intent phân loại/chẩn đoán"""
        if self.current_image is None:
            return "❌ No image available. Please upload an image first."
        
        triage = self.last_triage_result
        if not triage:
            return "⚠️ Image not yet analyzed. Please wait..."
        
        response = f"""📋 **Classification Results**

**Tool Used:** BiomedCLIP (Triage)

**Primary Classification:** {triage['modality']}
**Confidence:** {triage['confidence']:.1%}

**Top Classifications:**
"""
        sorted_preds = sorted(triage['all_predictions'].items(), key=lambda x: x[1], reverse=True)[:5]
        for label, prob in sorted_preds:
            bar = "█" * int(prob * 20)
            response += f"  • {label}: {prob:.1%} {bar}\n"
        
        return response
    
    def _handle_description_intent(self, user_message):
        """Xử lý intent mô tả"""
        if self.current_image is None:
            return "❌ No image available. Please upload an image first."
        
        triage = self.last_triage_result
        modality = triage['modality'] if triage else "medical image"
        
        response = f"""📝 **Image Description**

**Image Type:** {modality}
**Confidence:** {triage['confidence']:.1%} if triage else 'N/A'

This appears to be a {modality}. 

"""
        if "X-ray" in modality:
            response += """Based on the image characteristics:
- Standard radiographic imaging
- Shows bone and soft tissue contrast
- Commonly used for chest, skeletal, and abdominal imaging"""
        elif "MRI" in modality:
            response += """Based on the image characteristics:
- Magnetic resonance imaging
- Excellent soft tissue contrast
- Commonly used for brain, spine, and joint imaging"""
        elif "CT" in modality:
            response += """Based on the image characteristics:
- Computed tomography scan
- Cross-sectional imaging
- Good for detecting tumors, bleeding, and bone abnormalities"""
        
        return response
    
    def _handle_general_intent(self, user_message):
        """Xử lý các câu hỏi chung"""
        if self.current_image is None:
            return "👋 Hello! Please upload a medical image to start the analysis. I can help with:\n- 🔍 Detection of abnormalities\n- 🎯 Segmentation\n- 📋 Classification\n- 📝 Description"
        
        triage = self.last_triage_result
        return f"""I'm analyzing a {triage['modality'] if triage else 'medical image'}.

How can I help you? You can ask me to:
- **Detect** abnormalities or specific findings
- **Segment** regions of interest
- **Classify** the image type
- **Describe** what I see in the image

Just type your question!"""
    
    def chat(self, user_message, image=None):
        """
        Main chat function - xử lý tin nhắn từ user
        
        Args:
            user_message: Câu hỏi/tin nhắn của user
            image: PIL Image (optional, nếu user gửi ảnh mới)
        
        Returns:
            str: Response từ agent
        """
        # Nếu có ảnh mới, set và chạy triage
        if image is not None:
            self.set_image(image)
        
        # Nếu chưa có phiên, tạo mới
        if not self.session_active:
            self.start_new_session(self.current_image)
        
        # Inject context vào message
        if self.last_triage_result:
            enhanced_message = self.inject_context(user_message, verbose=False)
        else:
            enhanced_message = user_message
        
        # Lưu user message vào history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Generate response
        response = self._generate_response(user_message)
        
        # Lưu assistant response vào history
        self.conversation_history.append({
            "role": "assistant", 
            "content": response
        })
        
        return response
    
    def get_conversation_history(self):
        """Lấy lịch sử chat"""
        return self.conversation_history
    
    def clear_session(self):
        """Xóa phiên chat hiện tại"""
        self.conversation_history = []
        self.current_image = None
        self.last_triage_result = None
        self.detected_boxes = []
        self.verified_boxes = []
        self.session_active = False
        print("🗑️ Session cleared.")

# Initialize agent
agent = TriMedAgent(model, preprocess, tokenizer, TRIMED_CONFIG, device)
print("\n✅ TriMedAgent initialized with Chatbot support!")

## 5. Load Test Image

In [ ]:
# Create images folder if not exists
os.makedirs("images", exist_ok=True)

# Try to load a test image
test_image_paths = [
    "images/eval1.jpg",
    "images/test_xray.jpg",
    "images/chest_xray.png"
]

test_image = None
for path in test_image_paths:
    if os.path.exists(path):
        test_image = Image.open(path).convert("RGB")
        print(f"✅ Loaded image from: {path}")
        break

if test_image is None:
    # Download a sample chest X-ray from the internet
    print("⚠️ No local image found. Downloading sample chest X-ray...")
    try:
        # Use a public medical image
        url = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_Xray_PA_3-8-2010.png/440px-Chest_Xray_PA_3-8-2010.png"
        response = requests.get(url, timeout=10)
        test_image = Image.open(BytesIO(response.content)).convert("RGB")
        test_image.save("images/sample_chest_xray.png")
        print("✅ Downloaded and saved sample image.")
    except Exception as e:
        print(f"❌ Could not download image: {e}")
        print("Please place a medical image in the 'images' folder.")

if test_image:
    plt.figure(figsize=(8, 8))
    plt.imshow(test_image)
    plt.title(f"Test Image ({test_image.size[0]}x{test_image.size[1]})")
    plt.axis('off')
    plt.show()

## 6. Demo: Step 1 - Visual Triage

In [ ]:
if test_image:
    triage_result = agent.run_triage(test_image)

## 7. Demo: Step 2 - Context Injection

In [ ]:
if test_image:
    # Simulate a user question
    user_question = "Can you detect any abnormalities in this medical image? Please identify and localize any lesions or pathological findings."
    
    enhanced_prompt = agent.inject_context(user_question)

## 8. Demo: Step 3 - Gatekeeper Verification

In [ ]:
if test_image:
    w, h = test_image.size
    
    # Simulate bounding boxes from a detection model (normalized coords 0-1)
    # Box 1: Center region (likely pathology)
    # Box 2: Corner region (likely background/noise)
    # Box 3: Another region
    simulated_boxes = [
        [0.3, 0.3, 0.7, 0.7],   # Center - likely real finding
        [0.0, 0.0, 0.15, 0.15], # Top-left corner - likely noise
        [0.4, 0.2, 0.6, 0.5],   # Upper-center - possible finding
        [0.85, 0.85, 1.0, 1.0]  # Bottom-right corner - likely noise
    ]
    
    # Run gatekeeper
    target = "lung opacity" if "X-ray" in triage_result.get("modality", "") else "abnormality"
    gate_result = agent.run_gatekeeper(test_image, simulated_boxes, target)

## 9. Visualize Gatekeeper Results

In [ ]:
def visualize_gatekeeper(image, gate_result, title="Gatekeeper Verification"):
    """Visualize gatekeeper results with bounding boxes"""
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    w, h = image.size
    
    # Left: Before filtering (all boxes)
    axes[0].imshow(image)
    axes[0].set_title("Before Gatekeeper (All Detections)", fontsize=14)
    
    for detail in gate_result["details"]:
        box = detail["box"]
        x1, y1, x2, y2 = box[0]*w, box[1]*h, box[2]*w, box[3]*h
        color = "lime" if detail["passed"] else "red"
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 linewidth=3, edgecolor=color, facecolor='none')
        axes[0].add_patch(rect)
        
        label = f"Box {detail['box_index']}: {detail['confidence']:.0%}"
        axes[0].text(x1, y1-5, label, color=color, fontsize=10, weight='bold',
                    bbox=dict(facecolor='black', alpha=0.7, pad=2))
    
    axes[0].axis('off')
    
    # Right: After filtering (verified boxes only)
    axes[1].imshow(image)
    axes[1].set_title("After Gatekeeper (Verified Only)", fontsize=14)
    
    for i, box in enumerate(gate_result["verified_boxes"]):
        x1, y1, x2, y2 = box[0]*w, box[1]*h, box[2]*w, box[3]*h
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 linewidth=3, edgecolor='lime', facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x1, y1-5, f"Verified {i}", color='lime', fontsize=10, weight='bold',
                    bbox=dict(facecolor='black', alpha=0.7, pad=2))
    
    axes[1].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    print(f"\n📊 Summary:")
    print(f"   Before: {len(gate_result['details'])} boxes")
    print(f"   After:  {len(gate_result['verified_boxes'])} boxes")
    print(f"   Filtered out: {len(gate_result['rejected_indices'])} false positives")

if test_image and gate_result:
    visualize_gatekeeper(test_image, gate_result, 
                        f"Gatekeeper: {triage_result['modality']} Analysis")

## 10. Full Pipeline Demo

In [ ]:
if test_image:
    # Run the complete pipeline
    user_question = "Please analyze this medical image and detect any abnormalities or lesions."
    
    pipeline_result = agent.run_full_pipeline(
        image=test_image,
        user_question=user_question,
        simulated_boxes=simulated_boxes,
        target_entity="pathological finding"
    )
    
    # Final visualization
    if pipeline_result["gatekeeper"]:
        visualize_gatekeeper(test_image, pipeline_result["gatekeeper"],
                            f"Full Pipeline Result: {pipeline_result['triage']['modality']}")

## 11. Test with Different Images

In [ ]:
def test_with_url(url, description="Test Image"):
    """Test the pipeline with an image from URL"""
    try:
        print(f"\n🌐 Loading: {description}")
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        
        # Run triage only
        result = agent.run_triage(img)
        
        # Display
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.title(f"{description}\nDetected: {result['modality']} ({result['confidence']:.0%})")
        plt.axis('off')
        plt.show()
        
        return result
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Uncomment to test with different images:
# test_with_url("https://example.com/brain_mri.jpg", "Brain MRI")
# test_with_url("https://example.com/ct_scan.jpg", "CT Scan")

## 12. Configuration Inspection

In [ ]:
print("📋 Current TriMedAgent Configuration:\n")
print(json.dumps(TRIMED_CONFIG, indent=2, ensure_ascii=False))

---

## 🎯 Summary

TriMedAgent implements a **3-stage Intelligent Triage Pipeline**:

| Stage | Component | Function |
|-------|-----------|----------|
| **1. Perception** | BiomedCLIP | Classify image modality (X-ray, MRI, CT, etc.) |
| **2. Reasoning** | LLaVA-Med + Context | Enhanced prompts with modality context |
| **3. Gatekeeping** | BiomedCLIP | Filter false positives from detection |

### Key Features:
- ✅ **Configurable thresholds** via `labels.json`
- ✅ **Conditional execution** - skip unnecessary tools
- ✅ **Tool recommendations** based on modality
- ✅ **Synchronized filtering** - boxes, logits, phrases, masks

### Files Modified:
- `serve/labels.json` - Configuration
- `llava/serve/gradio_web_server_mmedagent.py` - Pipeline implementation

---

## 🤖 PART 2: CHATBOT INTERFACE

Phần này demo chức năng **Chatbot** - gửi ảnh và hỏi đáp về ảnh trong phiên chat.

### Key Features:
1. **Session Management**: Mỗi phiên chat có context riêng
2. **Auto Triage**: Tự động chạy 3 giai đoạn khi gửi ảnh
3. **Intent Detection**: Nhận diện intent từ câu hỏi
4. **Tool Routing**: Tự động chọn công cụ phù hợp

In [ ]:
# ============================================================
# 📱 CHATBOT DEMO - Console Version
# ============================================================

print("="*70)
print("🤖 TRIMEDAGENT CHATBOT DEMO (Console)")
print("="*70)

# Start new session với test image
agent.start_new_session(test_image)

# Simulate conversation
print("\n" + "-"*50)
print("💬 CONVERSATION DEMO")
print("-"*50)

# Turn 1: Ask about the image
response1 = agent.chat("What type of image is this?")
print(f"\n👤 User: What type of image is this?")
print(f"🤖 Agent: {response1[:500]}...")

# Turn 2: Ask for detection
response2 = agent.chat("Can you detect any abnormalities?")
print(f"\n👤 User: Can you detect any abnormalities?")
print(f"🤖 Agent: {response2[:500]}...")

# Turn 3: Ask for description
response3 = agent.chat("Please describe what you see")
print(f"\n👤 User: Please describe what you see")
print(f"🤖 Agent: {response3[:500]}...")

print("\n" + "="*70)
print("✅ Console chatbot demo completed!")
print("="*70)

### 🖥️ Gradio Chatbot UI

Giao diện chat interactive với Gradio - cho phép upload ảnh và chat realtime.

In [ ]:
# ============================================================
# 🎨 GRADIO CHATBOT UI
# ============================================================

import gradio as gr

class GradioChatbot:
    """Wrapper cho Gradio UI"""
    
    def __init__(self, agent):
        self.agent = agent
        self.current_image = None
    
    def upload_image(self, image):
        """Xử lý khi user upload ảnh"""
        if image is None:
            return "Please upload an image.", []
        
        self.current_image = image
        self.agent.start_new_session(image)
        
        triage = self.agent.last_triage_result
        welcome_msg = f"""🖼️ **Image Uploaded Successfully!**

**Triage Results (Stage 1):**
- **Type:** {triage['modality']}
- **Confidence:** {triage['confidence']:.1%}
- **Recommended Tools:** {', '.join(triage['recommended_tools'])}

You can now ask questions about this image. Try:
- "What do you see in this image?"
- "Detect any abnormalities"
- "Describe the findings"
"""
        # Return updated chat history
        return welcome_msg, [[None, welcome_msg]]
    
    def respond(self, message, chat_history):
        """Xử lý tin nhắn chat"""
        if not message:
            return "", chat_history
        
        # Get response from agent
        response = self.agent.chat(message, self.current_image if len(chat_history) == 0 else None)
        
        # Append to history
        chat_history.append([message, response])
        
        return "", chat_history
    
    def clear(self):
        """Clear chat"""
        self.agent.clear_session()
        self.current_image = None
        return None, [], "Session cleared. Upload a new image to start."
    
    def create_ui(self):
        """Tạo Gradio interface"""
        with gr.Blocks(title="TriMedAgent Chatbot", theme=gr.themes.Soft()) as demo:
            gr.Markdown("""
            # 🏥 TriMedAgent Medical Chatbot
            
            **3-Stage Intelligent Medical Image Analysis:**
            1. 🔬 **Perception** - Automatic image triage
            2. 🧠 **Reasoning** - Context-aware responses
            3. 🛡️ **Gatekeeping** - Verified detections
            
            Upload a medical image and start chatting!
            """)
            
            with gr.Row():
                with gr.Column(scale=1):
                    image_input = gr.Image(
                        label="📷 Upload Medical Image",
                        type="pil",
                        height=300
                    )
                    upload_btn = gr.Button("🚀 Analyze Image", variant="primary")
                    status = gr.Textbox(label="Status", value="Upload an image to start", interactive=False)
                    
                    gr.Markdown("""
                    ### 💡 Sample Questions:
                    - What type of image is this?
                    - Can you detect any abnormalities?
                    - Describe the findings
                    - Segment the detected regions
                    """)
                
                with gr.Column(scale=2):
                    chatbot = gr.Chatbot(
                        label="💬 Chat",
                        height=400,
                        bubble_full_width=False
                    )
                    
                    with gr.Row():
                        msg = gr.Textbox(
                            label="Your message",
                            placeholder="Ask about the medical image...",
                            scale=4
                        )
                        submit_btn = gr.Button("Send", variant="primary", scale=1)
                    
                    clear_btn = gr.Button("🗑️ Clear Chat")
            
            # Event handlers
            upload_btn.click(
                self.upload_image,
                inputs=[image_input],
                outputs=[status, chatbot]
            )
            
            msg.submit(
                self.respond,
                inputs=[msg, chatbot],
                outputs=[msg, chatbot]
            )
            
            submit_btn.click(
                self.respond,
                inputs=[msg, chatbot],
                outputs=[msg, chatbot]
            )
            
            clear_btn.click(
                self.clear,
                outputs=[image_input, chatbot, status]
            )
        
        return demo

# Create chatbot UI
print("🎨 Creating Gradio Chatbot UI...")
chatbot_ui = GradioChatbot(agent)
demo = chatbot_ui.create_ui()
print("✅ Gradio UI created!")

In [ ]:
# ============================================================
# 🚀 LAUNCH CHATBOT
# ============================================================

# Uncomment dòng dưới để chạy Gradio UI
# demo.launch(share=True)  # share=True để tạo public link

# Hoặc chạy local:
demo.launch(
    server_name="0.0.0.0",  # Cho phép truy cập từ network
    server_port=7860,
    share=False,  # Set True nếu muốn public link
    inbrowser=True  # Tự động mở browser
)

---

## 📊 Summary

### TriMedAgent Pipeline Flow:

```
User uploads image
        ↓
┌───────────────────────────────────────┐
│  STAGE 1: PERCEPTION (Triage)         │
│  BiomedCLIP classifies image type     │
│  → X-ray? MRI? CT? Pathology?        │
│  → Recommend appropriate tools        │
└───────────────────────────────────────┘
        ↓
User asks question
        ↓
┌───────────────────────────────────────┐
│  STAGE 2: REASONING (Context)         │
│  Inject triage context into prompt    │
│  → Enhanced medical understanding     │
└───────────────────────────────────────┘
        ↓
If detection/segmentation requested:
        ↓
┌───────────────────────────────────────┐
│  STAGE 3: GATEKEEPING (Verification)  │
│  BiomedCLIP verifies each detection   │
│  → Filter false positives             │
│  → Return only verified regions       │
└───────────────────────────────────────┘
        ↓
Return response to user
```

### Chatbot Features:
- **Session-based**: Mỗi phiên chat có context riêng
- **Auto-triage**: Tự động phân loại ảnh khi upload
- **Intent detection**: Nhận diện yêu cầu từ câu hỏi
- **Multi-turn**: Hỗ trợ nhiều lượt chat trong 1 phiên
- **Tool routing**: Tự động chọn tool phù hợp (DINO, MedSAM, etc.)

### Files Structure:
```
├── demo_trimedagent.ipynb    # This notebook
├── serve/labels.json         # Configuration & thresholds
├── llava/serve/gradio_web_server_mmedagent.py  # Main server
├── README_TRIMEDAGENT.md     # Full documentation
└── QUICKSTART.md             # Quick start guide
```